# Part2：AI Agentを使った分析

## 自動車保険の約款データを使ったRAGの構築


In [ ]:
%%sql -r dataframe_1
-- チャンク分割用の新しいテーブルの作成
CREATE OR REPLACE TABLE handson.car_insurance.parsed_terms_and_conditions_chunks (
    relative_path VARCHAR,
    file_url VARCHAR,
    title VARCHAR,
    CHUNK VARCHAR
);

-- PDFからパースしてチャンク分割
INSERT INTO handson.car_insurance.parsed_terms_and_conditions_chunks (relative_path, file_url, title, CHUNK)
SELECT
    d.relative_path,
    d.file_url,
    REGEXP_SUBSTR(d.relative_path, '[^/]+$') AS title,
    c.value AS CHUNK
FROM
    DIRECTORY(@handson.car_insurance.pdf) d,
    LATERAL FLATTEN( input => SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER (
        SNOWFLAKE.CORTEX.PARSE_DOCUMENT(
            @handson.car_insurance.pdf,
            d.relative_path,
            {'mode': 'LAYOUT'}
        ):content,
        'markdown',
        2500,
        500
    )) c
WHERE d.relative_path = 'driver_insurance_yakkan.pdf';

In [ ]:
%%sql -r dataframe_11
-- チャンク化したデータの確認
SELECT * FROM handson.car_insurance.parsed_terms_and_conditions_chunks;

In [ ]:
%%sql -r dataframe_12
-- Cortex Search Serviceの作成
CREATE OR REPLACE CORTEX SEARCH SERVICE handson.car_insurance.terms_and_conditions_service
    ON chunk
    ATTRIBUTES relative_path, file_url, title
    WAREHOUSE = compute_wh
    TARGET_LAG = '30 day'
    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (
        SELECT
            relative_path,
            file_url,
            title,
            chunk
        FROM handson.car_insurance.parsed_terms_and_conditions_chunks
    );

In [ ]:
%%sql -r dataframe_13
-- 動くか確認
SELECT 
    '事故対応手順検索' as search_type,
    (results.index + 1) as result_rank,
    results.value['title']::string as document_title,
    results.value['chunk']::string as content,
    LENGTH(results.value['chunk']::string) as content_length,
    -- results.value['file_url']::string as source_file_url
FROM (
    SELECT 
        PARSE_JSON(
            SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
                'handson.car_insurance.terms_and_conditions_service',
                '{
                    "query": "事故が発生した場合の対応手順を教えてください。",
                    "columns": ["chunk", "title", "file_url"],
                    "limit": 5
                }'
            )
        )['results'] as results_array
),
LATERAL FLATTEN(input => results_array) as results
ORDER BY results.index;

### 発展：Cortex Search ダウンロードリンク生成ツール
- Cortex Searchで検索したドキュメントのダウンロードリンクを生成するストアドプロシージャを用意
- 最終的にSnowflake Intelligenceを使ってソースとなるドキュメントのダウンロードが可能に

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE PROCEDURE handson.car_insurance.get_file_presigned_url_sp(
RELATIVE_FILE_PATH STRING, 
EXPIRATION_MINS INTEGER DEFAULT 60
)
RETURNS STRING
LANGUAGE SQL
COMMENT = '@handson.car_insurance.pdf内のファイルに対するpresigned URLを生成します。入力は相対ファイルパスです。'
EXECUTE AS CALLER
AS
$$
DECLARE
    presigned_url STRING;
    sql_stmt STRING;
    expiration_seconds INTEGER;
    stage_name STRING DEFAULT '@handson.car_insurance.pdf';
BEGIN
    expiration_seconds := EXPIRATION_MINS * 60;
    
    sql_stmt := 'SELECT GET_PRESIGNED_URL(' || stage_name || ', ' || '''' || RELATIVE_FILE_PATH || '''' || ', ' || expiration_seconds || ') AS url';

    EXECUTE IMMEDIATE :sql_stmt;

    SELECT "URL"
    INTO :presigned_url
    FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

    RETURN :presigned_url;
END;
$$;

## 自動車保険システムからセマンティックビューの作成
### セマンティックビューの定義

#### ファクト：ファクトは分析の最終的な指標そのものではなく、その指標を計算するための基礎となる構成要素

#### メトリクス：複数の行にわたって同じテーブルのファクトや他の列を（SUM、AVG、COUNT などの関数を使用して）集約することによって計算される、ビジネス・パフォーマンスの定量化可能な尺度
例) 総収入や利益率など。メトリクスは、ビジネスの意思決定を促進するレポートやダッシュボードの KPI を表します。
#### ディメンション：データをグループ化したり、絞り込んだり、分析したりするための「切り口」や「視点」を提供するもの

例) 購入日、顧客の詳細、製品カテゴリー、場所など、「誰が」「何を」「どこで」「いつ」問題に答えるもの

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE SEMANTIC VIEW handson.car_insurance.insurance_claims_semantic_view
    TABLES (
        CLAIMS AS handson.car_insurance.claims PRIMARY KEY (CLAIM_ID) 
            WITH SYNONYMS = ('保険金請求','請求','事故') 
            COMMENT = '保険金請求に関する中心的なファクトテーブル',
        POLICIES AS handson.car_insurance.policies PRIMARY KEY (POLICY_ID) 
            WITH SYNONYMS = ('保険契約','契約','ポリシー') 
            COMMENT = '保険契約者と車両に関するディメンションテーブル',
        REPAIR_SHOPS AS handson.car_insurance.repair_shops PRIMARY KEY (SHOP_ID) 
            WITH SYNONYMS = ('修理工場','工場','パートナー工場') 
            COMMENT = '提携している修理工場に関するディメンションテーブル',
        REPAIR_ORDERS AS handson.car_insurance.repair_orders PRIMARY KEY (ORDER_ID) 
            WITH SYNONYMS = ('修理指示','修理明細','作業指示') 
            COMMENT = '各請求に対する修理作業の明細データ'
    )
    RELATIONSHIPS (
        CLAIMS_TO_POLICIES AS CLAIMS(POLICY_ID) REFERENCES POLICIES(POLICY_ID),
        CLAIMS_TO_SHOPS AS CLAIMS(REPAIR_SHOP_ID) REFERENCES REPAIR_SHOPS(SHOP_ID),
        ORDERS_TO_CLAIMS AS REPAIR_ORDERS(CLAIM_ID) REFERENCES CLAIMS(CLAIM_ID)
    )
    FACTS (
        CLAIMS.claim_record AS 1 
            COMMENT = '請求レコード数。件数のカウントに使用',
        CLAIMS.approved_cost_yen AS APPROVED_COST_YEN 
            COMMENT = '承認された修理費用（円）',
        CLAIMS.estimate_cost_yen AS ESTIMATE_COST_YEN 
            COMMENT = '見積もり修理費用（円）',
        CLAIMS.paid_to_date_yen AS PAID_TO_DATE_YEN 
            COMMENT = '支払済み金額（円）',
        POLICIES.policy_record AS 1 
            COMMENT = '契約レコード数。件数のカウントに使用',
        POLICIES.annual_premium_yen AS ANNUAL_PREMIUM_YEN 
            COMMENT = '年間保険料（円）',
        REPAIR_ORDERS.repair_cost AS LINE_TOTAL_AFTER_ADJ_YEN
            COMMENT = '修理明細の合計金額（調整後、円）',
        REPAIR_ORDERS.labor_hours AS LABOR_HOURS
            COMMENT = '作業時間（時間）'
    )
    DIMENSIONS (
        CLAIMS.incident_date AS INCIDENT_DATE 
            WITH SYNONYMS = ('事故日','発生日') 
            COMMENT = '事故が発生した日付',
        CLAIMS.incident_year AS YEAR(incident_date) 
            WITH SYNONYMS = ('事故年','発生年') 
            COMMENT = '事故が発生した年',
        CLAIMS.incident_month AS MONTH(incident_date) 
            WITH SYNONYMS = ('事故月','発生月') 
            COMMENT = '事故が発生した月',
        CLAIMS.claim_status AS CLAIM_STATUS 
            WITH SYNONYMS = ('請求ステータス','ステータス') 
            COMMENT = '保険金請求の現在のステータス (Open, Approved, Paid, Denied)',
        CLAIMS.loss_cause AS LOSS_CAUSE 
            WITH SYNONYMS = ('事故原因','原因') 
            COMMENT = '損害の原因 (Collision, Weather, Theftなど)',
        CLAIMS.severity AS SEVERITY 
            WITH SYNONYMS = ('損害の大きさ','深刻度') 
            COMMENT = '損害の深刻度 (Minor, Moderate, Severe, Total Loss)',
        CLAIMS.incident_prefecture AS INCIDENT_PREFECTURE 
            WITH SYNONYMS = ('事故発生都道府県','事故現場の都道府県') 
            COMMENT = '事故が発生した都道府県',
        CLAIMS.fraud_score AS FRAUD_SCORE
            WITH SYNONYMS = ('不正スコア', '詐欺スコア')
            COMMENT = '請求の不正行為の可能性を示すスコア',
        POLICIES.vehicle_make AS VEHICLE_MAKE 
            WITH SYNONYMS = ('自動車メーカー','メーカー') 
            COMMENT = '車両のメーカー',
        POLICIES.vehicle_model AS VEHICLE_MODEL
            WITH SYNONYMS = ('車種','モデル')
            COMMENT = '車両のモデル名',
        POLICIES.vehicle_age_band AS VEHICLE_AGE_BAND 
            WITH SYNONYMS = ('車齢帯','車両年式') 
            COMMENT = '車両の年式帯 (Newer, Mid, Older)',
        POLICIES.vehicle_use AS VEHICLE_USE
            WITH SYNONYMS = ('車両用途','使用目的')
            COMMENT = '車両の主な使用目的 (Personal, Commute, Commercial)',
        POLICIES.driver_age AS DRIVER_AGE
            WITH SYNONYMS = ('運転者年齢','ドライバー年齢')
            COMMENT = '契約時の運転者の年齢',
        POLICIES.policy_status AS POLICY_STATUS 
            WITH SYNONYMS = ('契約ステータス') 
            COMMENT = '保険契約のステータス (Active, Lapsed, Cancelled)',
        POLICIES.registered_prefecture AS REGISTERED_PREF
            WITH SYNONYMS = ('登録都道府県')
            COMMENT = '車両が登録されている都道府県',
        REPAIR_SHOPS.shop_name AS SHOP_NAME 
            WITH SYNONYMS = ('修理工場名','工場名') 
            COMMENT = '修理工場の名称',
        REPAIR_SHOPS.shop_prefecture AS PREFECTURE 
            WITH SYNONYMS = ('工場都道府県','工場所在地') 
            COMMENT = '修理工場の都道府県',
        REPAIR_SHOPS.partner_tier AS PARTNER_TIER 
            WITH SYNONYMS = ('パートナーランク','ティア') 
            COMMENT = '修理工場のパートナーランク (A, B, C)',
        REPAIR_SHOPS.shop_rating AS RATING
            WITH SYNONYMS = ('工場評価','評価')
            COMMENT = '修理工場の評価スコア',
        REPAIR_ORDERS.work_start_date AS WORK_START_DATE
            WITH SYNONYMS = ('修理開始日')
            COMMENT = '修理作業の開始日',
        REPAIR_ORDERS.work_end_date AS WORK_END_DATE
            WITH SYNONYMS = ('修理完了日')
            COMMENT = '修理作業の完了日'
    )
    METRICS (
        CLAIMS.TOTAL_CLAIMS AS COUNT(claims.claim_record) 
            WITH SYNONYMS = ('総請求件数','合計請求件数')
            COMMENT = 'すべての保険金請求の合計件数',
        CLAIMS.TOTAL_APPROVED_COST AS SUM(claims.approved_cost_yen) 
            WITH SYNONYMS = ('総承認費用','合計承認費用')
            COMMENT = '承認された修理費用の合計額',
        CLAIMS.AVERAGE_APPROVED_COST AS AVG(claims.approved_cost_yen) 
            WITH SYNONYMS = ('平均承認費用')
            COMMENT = '承認された修理費用の平均額',
        CLAIMS.TOTAL_PAID_AMOUNT AS SUM(claims.paid_to_date_yen)
            WITH SYNONYMS = ('総支払額','合計支払額')
            COMMENT = '支払われた保険金の合計額',
        POLICIES.TOTAL_POLICIES AS COUNT(policies.policy_record) 
            WITH SYNONYMS = ('総契約件数','合計契約件数')
            COMMENT = '保険契約の総件数',
        POLICIES.TOTAL_PREMIUM_REVENUE AS SUM(policies.annual_premium_yen) 
            WITH SYNONYMS = ('総保険料収入','合計保険料')
            COMMENT = '年間保険料の合計額',
        REPAIR_ORDERS.TOTAL_REPAIR_COST AS SUM(repair_orders.repair_cost)
            WITH SYNONYMS = ('総修理費用','合計修理費用')
            COMMENT = '修理にかかった費用の総額'
    )
    COMMENT = '自動車保険の契約、請求、修理に関する分析のためのセマンティックビュー';

## 各テーブルのディスクリプションやカラムの説明を自動生成
- AI_GENERATE_TABLE_DESC関数を使い、テーブルや列のディスクリプションを自動作成
- 現時点では、英語しか対応していないので注意

In [ ]:
%%sql -r dataframe_5
{% raw %}
-- AI_GENERATE_TABLE_DESCで説明を生成し、テーブル・カラムのCOMMENTに自動反映するプロシージャ
CREATE OR REPLACE PROCEDURE HANDSON.CAR_INSURANCE.AI_TABLE_DESC(table_name STRING)
  RETURNS STRING
  LANGUAGE PYTHON
  RUNTIME_VERSION = '3.10'
  PACKAGES = ('snowflake-snowpark-python')
  HANDLER = 'main'
  EXECUTE AS CALLER
AS
$$
import json

def main(session, table_name):
    result = session.sql(f"""
        CALL AI_GENERATE_TABLE_DESC(
            '{table_name}',
            {{'describe_columns': true, 'use_table_data': true}}
        )
    """).collect()

    output = json.loads(result[0][0])

    table_desc = output["TABLE"][0]["description"].replace("'", "\\'")
    db = output["TABLE"][0]["database_name"]
    sch = output["TABLE"][0]["schema_name"]
    tbl = output["TABLE"][0]["name"]
    fqn = f"{db}.{sch}.{tbl}"

    session.sql(f"ALTER TABLE {fqn} SET COMMENT = '{table_desc}'").collect()

    for col in output.get("COLUMNS", []):
        col_desc = col["description"].replace("'", "\\'")
        col_name = col["name"]
        if not col_name.isupper():
            col_name = f'"{ col_name}"'
        session.sql(f"ALTER TABLE {fqn} MODIFY COLUMN {col_name} COMMENT '{col_desc}'").collect()

    return 'Success'
$$;
{% endraw %}

-- 実行：各テーブルに説明を生成・反映
CALL HANDSON.CAR_INSURANCE.AI_TABLE_DESC('HANDSON.CAR_INSURANCE.INSURANCE_FEEDBACK');

### Cortex Codeを使って、以下をやってみましょう
- AI_TRANSLATE関数を使って、テーブルや列のディスクリプションを日本語にする
- 他のテーブルも同じようにテーブルや列のディスクリプションを自動作成する

## AI Agentの作成→チャットbotの作成
Cortex Agentを作成したのち、Snowflake IntelligenceはSnowsight（UI）で設定します

In [ ]:
%%sql -r dataframe_8
-- Cortex AgentをSQLで作成するクエリ
CREATE OR REPLACE AGENT snowflake_intelligence.agents.driver_insurance_chatbot_agent
WITH PROFILE='{ "display_name": "driver_insurance_agent" }'
    COMMENT= '自動車保険に関する質問に回答するエージェントです。'

FROM SPECIFICATION $$
{
  "models": {
    "orchestration": ""
  },
  "instructions": {
    "response": "あなたは自動車保険に関するのデータマートにアクセスできるデータアナリストです。ユーザーが日付範囲を指定しない場合は、2025年と仮定してください。すべてのドメインからのデータを活用して、ユーザーの質問を分析し回答してください。可能であれば視覚化を提供してください。トレンドラインは線グラフをデフォルトとし、カテゴリは棒グラフを使用してください。",
    "orchestration": "既知のエンティティに対してcortex searchを使用し、詳細な分析のためにcortex analystに結果を渡してください。",
    "sample_questions": [
      {
        "question": "事故が発生した都道府県ごとに、平均承認費用を比較して、高い順に並べて可視化してください"
      }
    ]
  },
  "tools": [
    {
      "tool_spec": {
        "type": "cortex_analyst_text_to_sql",
        "name": "DRIVER INSURANCE SYSTEM DATAMART",
        "description": "お客様の自動車保険に関するデータをクエリすることをユーザーに許可します。"
      }
    },
    {
      "tool_spec": {
        "type": "cortex_search",
        "name": "DRIVER INSURANCE SEARCH",
        "description": "このツールはドライバー保険の約款データを検索するために使用します。"
      }
    },
    {
      "tool_spec": {
        "type": "generic",
        "name": "Doc URL Tool",
        "description": "このツールは、参照ドキュメント用のCortex Searchツールから取得したrelative_pathを使用し、ユーザーがドキュメントを表示・ダウンロードするための一時URLを返します。\n\n返されたURLは、ドキュメントタイトルをテキストとし、このツールの出力をURLとするHTMLハイパーリンクとして表示する必要があります。\n\nURLにPDFが含まれていないPDFドキュメントのURL形式は次のようになります。ファイルをダウンロードする代わりにブラウザでPDFドキュメントが開くようにハイパーリンク形式を作成してください。\nhttps://domain/path/unique_guid",
        "input_schema": {
          "type": "object",
          "properties": {
            "expiration_mins": {
              "description": "デフォルトは5にする必要があります",
              "type": "number"
            },
            "relative_file_path": {
              "description": "これはCortex Searchツールから取得されるrelative_pathの値です。",
              "type": "string"
            }
          },
          "required": [
            "expiration_mins",
            "relative_file_path"
          ]
        }
      }
    }
  ],
  "tool_resources": {
    "DRIVER INSURANCE SYSTEM DATAMART": {
      "semantic_view": "handson.car_insurance.insurance_claims_semantic_view"
    },
    "DRIVER INSURANCE SEARCH": {
      "id_column": "relative_path",
      "max_results": 5,
      "name": "handson.car_insurance.terms_and_conditions_service",
      "title_column": "title"
    },
    "Doc URL Tool": {
      "execution_environment": {
        "query_timeout": 0,
        "type": "warehouse",
        "warehouse": "COMPUTE_WH"
      },
      "identifier": "handson.car_insurance.get_file_presigned_url_sp",
      "name": "GET_FILE_PRESIGNED_URL_SP(VARCHAR, DEFAULT NUMBER)",
      "type": "procedure"
    },
  }
}
$$;

## 補足
- AI_GENERATE_TABLE_DESCの日本語対応版

In [ ]:
%%sql -r dataframe_4
-- AI_GENERATE_TABLE_DESCの日本語対応版
CREATE OR REPLACE PROCEDURE HANDSON.CAR_INSURANCE.AI_TABLE_DESC(table_name STRING)
  RETURNS STRING
  LANGUAGE PYTHON
  RUNTIME_VERSION = '3.10'
  PACKAGES = ('snowflake-snowpark-python')
  HANDLER = 'main'
  EXECUTE AS CALLER
AS
$$
import json

def main(session, table_name):
    result = session.sql(f"""
        CALL AI_GENERATE_TABLE_DESC(
            '{table_name}',
            {{'describe_columns': true, 'use_table_data': true}}
        )
    """).collect()

    output = json.loads(result[0][0])

    table_desc_en = output["TABLE"][0]["description"]
    table_desc = session.sql(f"SELECT AI_TRANSLATE('{table_desc_en.replace(chr(39), chr(39)+chr(39))}', 'en', 'ja')").collect()[0][0].replace("'", "\\'")
    db = output["TABLE"][0]["database_name"]
    sch = output["TABLE"][0]["schema_name"]
    tbl = output["TABLE"][0]["name"]
    fqn = f"{db}.{sch}.{tbl}"

    session.sql(f"ALTER TABLE {fqn} SET COMMENT = '{table_desc}'").collect()

    for col in output.get("COLUMNS", []):
        col_desc_en = col["description"]
        col_desc = session.sql(f"SELECT AI_TRANSLATE('{col_desc_en.replace(chr(39), chr(39)+chr(39))}', 'en', 'ja')").collect()[0][0].replace("'", "\\'")
        col_name = col["name"]
        if not col_name.isupper():
            col_name = f'"{col_name}"'
        session.sql(f"ALTER TABLE {fqn} MODIFY COLUMN {col_name} COMMENT '{col_desc}'").collect()

    return 'Success'
$$;

-- 実行：INSURANCE_FEEDBACKテーブルに説明を生成・反映
CALL HANDSON.CAR_INSURANCE.AI_TABLE_DESC('HANDSON.CAR_INSURANCE.INSURANCE_FEEDBACK');